# Multi-Voice Activity Detection — Single File Analysis

This notebook runs both the **Traditional (signal-processing)** and **DNN** MVAD models on a single WAV file and displays prediction waveforms for comparison.

In [ ]:
import sys
import numpy as np
import soundfile as sf
import scipy.io

# Import everything from mvad_test.py
# (mvad_test.py forces matplotlib backend to 'Agg' at import time,
#  so we must re-set it to 'inline' AFTER the import)
from mvad_test import (
    MultivoiceVAD,
    load_dnn_model,
    dnn_predict_file,
)

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
%matplotlib inline
import matplotlib.pyplot as plt

print('Imports OK, matplotlib backend:', matplotlib.get_backend())

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# INPUT_WAV   = 'dumps/example_3_SHU1300__TMR_cfg_single_talker_mobile_with_bbldagc_48kHz_txfeOut.wav'
INPUT_WAV   = 'inputs/example_12.wav'
GT_MAT      = 'inputs/example_12_manualVAD.mat'   # ground truth (sample-level)

# ── DNN model selection ───────────────────────────────────────────────────
# DNN_TYPE: 'mel'  → original mel-feature DNN (mvad_dnn_model_ep47.pt)
#           'i'    → VAD_I_C2_L8 raw-waveform 1D conv (mvad_dnn_i_model_ep25.pt)
#           'h'    → VAD_H_C24_L10 raw-waveform 1D conv (mvad_dnn_h_model.pt)
DNN_TYPE    = 'h'
DNN_MODEL_MEL = 'mvad_dnn_model_ep47.pt'
DNN_MODEL_I   = 'mvad_dnn_i_model_ep25.pt'
DNN_MODEL_H   = 'mvad_dnn_h_model_ep31.pt'
DNN_MODEL     = {'mel': DNN_MODEL_MEL, 'i': DNN_MODEL_I, 'h': DNN_MODEL_H}[DNN_TYPE]

# ── Traditional VAD parameters (defaults from mvad_test.py) ───────────────
FRAME_MS              = 30
HOP_MS                = 10
ENERGY_THRESHOLD_DB   = -40
PITCH_CONF_THRESH     = 0.25
YIN_THRESHOLD         = 0.15
SECONDARY_PITCH_CONF  = 0.20
SF_THRESHOLD          = 0.30
OVERLAP_THRESHOLD     = 0.38
CONTEXT_FRAMES        = 3
MEDIAN_FILTER         = 7
F0_MIN                = 80
F0_MAX                = 400

## 2. Load audio

In [ ]:
signal, sr = sf.read(INPUT_WAV, dtype='float64')
if signal.ndim > 1:
    signal = np.mean(signal, axis=1)

duration = len(signal) / sr
print(f'File     : {INPUT_WAV}')
print(f'SR       : {sr} Hz')
print(f'Duration : {duration:.2f} s')
print(f'Samples  : {len(signal):,}')

# ── Load ground truth ────────────────────────────────────────────────────
gt_data = scipy.io.loadmat(GT_MAT)
gt_vad  = gt_data['vad'].flatten()   # sample-level: 0=silence, 1=single, 2=overlap
print(f'GT MAT   : {GT_MAT}  ({len(gt_vad):,} samples)')
for lab, name in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
    cnt = np.sum(gt_vad == lab)
    print(f'  {name:20s}: {cnt/sr:.2f} s  ({cnt/len(gt_vad)*100:.1f}%)')

## 3. Run Traditional VAD

In [ ]:
vad = MultivoiceVAD(
    sr=sr,
    frame_ms=FRAME_MS,
    hop_ms=HOP_MS,
    energy_threshold_db=ENERGY_THRESHOLD_DB,
    pitch_conf_thresh=PITCH_CONF_THRESH,
    yin_threshold=YIN_THRESHOLD,
    secondary_pitch_conf=SECONDARY_PITCH_CONF,
    spectral_flatness_overlap=SF_THRESHOLD,
    overlap_threshold=OVERLAP_THRESHOLD,
    context_frames=CONTEXT_FRAMES,
    median_filter_size=MEDIAN_FILTER,
    f0_min=F0_MIN,
    f0_max=F0_MAX,
)

trad_labels, trad_features = vad.process(signal)

n_trad = len(trad_labels)
hop_s = vad.hop_len / sr
trad_times = np.arange(n_trad) * hop_s

print(f'Traditional VAD: {n_trad} frames')
for lab, name in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
    cnt = np.sum(trad_labels == lab)
    print(f'  {name:20s}: {cnt:6d} frames  ({cnt/n_trad*100:5.1f}%)  {cnt*hop_s:.2f}s')

## 4. Run DNN Model

In [ ]:
# ── VAD_I_C2_L8 model definition & helpers (for DNN_TYPE == 'i') ──────────
import torch
import torch.nn as nn
from math import gcd
from scipy.signal import resample_poly

TARGET_SAMPLE_RATE_I = 16_000
DECIMATION_FACTOR_I  = 128       # total network decimation (2^7)
NUM_CLASSES_I        = 3


class ResCbr1dGen(nn.Module):
    """Residual Conv-BatchNorm-ReLU 1D block (valid convolution, no padding)."""
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 residual=False):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              stride=stride, padding=0, bias=False)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.use_residual = residual and (in_channels == out_channels)
        self.stride = stride

    def forward(self, x):
        out = self.relu(self.bn(self.conv(x)))
        if self.use_residual:
            skip = x
            if self.stride > 1:
                skip = skip[:, :, ::self.stride]
            out_len = out.size(2)
            skip = skip[:, :, -out_len:]
            out = out + skip
        return out


class VAD_I_C2_L8(nn.Module):
    """1D Conv Encoder-Decoder for VAD (from vad_i_c2_l8_architecture.html)."""
    def __init__(self, num_classes=NUM_CLASSES_I):
        super().__init__()
        self.fwd = nn.ModuleList([
            ResCbr1dGen(1, 2, kernel_size=7, stride=2, residual=False),
            ResCbr1dGen(2, 4, kernel_size=7, stride=2, residual=False),
            ResCbr1dGen(4, 8, kernel_size=7, stride=2, residual=False),
        ])
        self.core = nn.ModuleList([
            ResCbr1dGen(8, 8, kernel_size=7, stride=2, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=2, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=2, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=2, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=1, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=1, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=1, residual=True),
        ])
        self.inv = nn.ModuleList([
            ResCbr1dGen(8, 4, kernel_size=9, stride=1, residual=False),
            ResCbr1dGen(4, num_classes, kernel_size=128, stride=1, residual=False),
        ])

    def forward(self, x):
        for layer in self.fwd:
            x = layer(x)
        for layer in self.core:
            x = layer(x)
        for layer in self.inv:
            x = layer(x)
        return x


def _resample_to_16k(audio, orig_sr):
    """Resample audio to 16 kHz."""
    if orig_sr == TARGET_SAMPLE_RATE_I:
        return audio.astype(np.float32)
    g = gcd(int(TARGET_SAMPLE_RATE_I), int(orig_sr))
    up = int(TARGET_SAMPLE_RATE_I) // g
    down = int(orig_sr) // g
    return resample_poly(audio, up, down).astype(np.float32)


def load_dnn_i_model(model_path, device=None):
    """Load a trained VAD_I_C2_L8 model from checkpoint."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = torch.load(str(model_path), map_location=device, weights_only=False)
    cfg = ckpt['config']
    model = VAD_I_C2_L8(num_classes=cfg.get('num_classes', NUM_CLASSES_I))
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  DNN-I model loaded: arch={cfg["arch"]}, '
          f'decimation={cfg["decimation_factor"]}×, '
          f'params={n_params:,}, device={device}')
    if 'epoch' in ckpt:
        print(f'  Best epoch: {ckpt["epoch"]}')
    return model, cfg, device


def dnn_i_predict_file(audio, orig_sr, model, cfg, device):
    """
    Run VAD_I_C2_L8 inference on a full audio file.

    The audio is resampled to 16 kHz, then processed in overlapping chunks.
    Returns per-frame predictions at 125 Hz (8 ms hop).
    """
    audio_16k = _resample_to_16k(audio, orig_sr)
    dec = cfg.get('decimation_factor', DECIMATION_FACTOR_I)
    chunk_samples = cfg.get('chunk_samples', 48_000)

    # Compute output length for this chunk size (analytically)
    LAYER_DEFS = [
        (7, 2), (7, 2), (7, 2),
        (7, 2), (9, 2), (9, 2), (9, 2),
        (9, 1), (9, 1), (9, 1),
        (9, 1), (128, 1),
    ]
    out_len = chunk_samples
    for k, s in LAYER_DEFS:
        out_len = (out_len - k) // s + 1

    stride_samples = out_len * dec
    total_frames = len(audio_16k) // dec

    predictions = np.full(total_frames, -1, dtype=np.int32)

    with torch.no_grad():
        for start in range(0, len(audio_16k) - chunk_samples + 1,
                           stride_samples):
            chunk = audio_16k[start: start + chunk_samples]
            x = torch.from_numpy(chunk).float().unsqueeze(0).unsqueeze(0)
            x = x.to(device)                              # (1, 1, chunk_samples)
            logits = model(x)                              # (1, C, out_len)
            preds = logits.argmax(dim=1).squeeze(0).cpu().numpy()  # (out_len,)

            # Right-aligned: output frames correspond to the rightmost part
            end_sample = start + chunk_samples
            end_frame = end_sample // dec
            frame_start = end_frame - out_len
            lo = max(0, frame_start)
            hi = min(total_frames, end_frame)
            src_lo = lo - frame_start
            src_hi = src_lo + (hi - lo)
            predictions[lo:hi] = preds[src_lo:src_hi]

    # Fill any remaining frames (if audio shorter than one chunk)
    if np.any(predictions < 0):
        # Process the tail with the last possible chunk
        if len(audio_16k) >= chunk_samples:
            start = len(audio_16k) - chunk_samples
            chunk = audio_16k[start: start + chunk_samples]
            x = torch.from_numpy(chunk).float().unsqueeze(0).unsqueeze(0).to(device)
            logits = model(x)
            preds = logits.argmax(dim=1).squeeze(0).cpu().numpy()
            end_frame = len(audio_16k) // dec
            frame_start = end_frame - out_len
            lo = max(0, frame_start)
            hi = min(total_frames, end_frame)
            src_lo = lo - frame_start
            src_hi = src_lo + (hi - lo)
            mask = predictions[lo:hi] < 0
            predictions[lo:hi] = np.where(mask, preds[src_lo:src_hi],
                                          predictions[lo:hi])
        # Any still unfilled → silence
        predictions[predictions < 0] = 0

    return predictions


# ── VAD_H_C24_L10 model definition & helpers (for DNN_TYPE == 'h') ─────────

TARGET_SAMPLE_RATE_H = 16_000
DECIMATION_FACTOR_H  = 64        # total network decimation (2^6)
NUM_CLASSES_H        = 3

LAYER_DEFS_H = [
    (128, 2), (9, 2), (9, 2), (9, 2), (9, 2),           # fwd L1-L5
    (9, 2),                                                # core L6 (stride-2)
    (9, 1), (9, 1), (9, 1), (9, 1), (9, 1), (9, 1),     # core L7-L12 (stride-1)
    (9, 1), (9, 1), (9, 1),                               # inv L13-L15
    (128, 1),                                              # inv L16
]


class VAD_H_C24_L10(nn.Module):
    """1D Conv Encoder-Decoder for VAD (from vad_h_c24_l10_architecture.html).

    16 layers, 64× decimation (250 Hz output), mixed alignment (center L1-L3, right L4-L16).
    Channels: 1→2→4→8→16→32 (encoder) → 32 (core) → 16→8→4→C (decoder).
    """
    def __init__(self, num_classes=NUM_CLASSES_H):
        super().__init__()
        self.fwd = nn.ModuleList([
            ResCbr1dGen(1,  2,  kernel_size=128, stride=2, residual=False),  # L1
            ResCbr1dGen(2,  4,  kernel_size=9,   stride=2, residual=False),  # L2
            ResCbr1dGen(4,  8,  kernel_size=9,   stride=2, residual=False),  # L3
            ResCbr1dGen(8,  16, kernel_size=9,   stride=2, residual=False),  # L4
            ResCbr1dGen(16, 32, kernel_size=9,   stride=2, residual=False),  # L5
        ])
        self.core = nn.ModuleList([
            ResCbr1dGen(32, 32, kernel_size=9, stride=2, residual=True),   # L6
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L7
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L8
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L9
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L10
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L11
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L12
        ])
        self.inv = nn.ModuleList([
            ResCbr1dGen(32, 16,         kernel_size=9,   stride=1, residual=False),  # L13
            ResCbr1dGen(16, 8,          kernel_size=9,   stride=1, residual=False),  # L14
            ResCbr1dGen(8,  4,          kernel_size=9,   stride=1, residual=False),  # L15
            ResCbr1dGen(4,  num_classes, kernel_size=128, stride=1, residual=False), # L16
        ])

    def forward(self, x):
        for layer in self.fwd:
            x = layer(x)
        for layer in self.core:
            x = layer(x)
        for layer in self.inv:
            x = layer(x)
        return x


def load_dnn_h_model(model_path, device=None):
    """Load a trained VAD_H_C24_L10 model from checkpoint."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = torch.load(str(model_path), map_location=device, weights_only=False)
    cfg = ckpt['config']
    model = VAD_H_C24_L10(num_classes=cfg.get('num_classes', NUM_CLASSES_H))
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  DNN-H model loaded: arch={cfg["arch"]}, '
          f'decimation={cfg["decimation_factor"]}×, '
          f'params={n_params:,}, device={device}')
    if 'epoch' in ckpt:
        print(f'  Best epoch: {ckpt["epoch"]}')
    return model, cfg, device


def dnn_h_predict_file(audio, orig_sr, model, cfg, device):
    """
    Run VAD_H_C24_L10 inference on a full audio file.

    The audio is resampled to 16 kHz, then processed in overlapping chunks.
    Returns per-frame predictions at 250 Hz (4 ms hop).
    """
    audio_16k = _resample_to_16k(audio, orig_sr)
    dec = cfg.get('decimation_factor', DECIMATION_FACTOR_H)
    chunk_samples = cfg.get('chunk_samples', 48_000)

    # Compute output length analytically
    out_len = chunk_samples
    for k, s in LAYER_DEFS_H:
        out_len = (out_len - k) // s + 1

    stride_samples = out_len * dec
    total_frames = len(audio_16k) // dec

    predictions = np.full(total_frames, -1, dtype=np.int32)

    with torch.no_grad():
        for start in range(0, len(audio_16k) - chunk_samples + 1,
                           stride_samples):
            chunk = audio_16k[start: start + chunk_samples]
            x = torch.from_numpy(chunk).float().unsqueeze(0).unsqueeze(0)
            x = x.to(device)                              # (1, 1, chunk_samples)
            logits = model(x)                              # (1, C, out_len)
            preds = logits.argmax(dim=1).squeeze(0).cpu().numpy()  # (out_len,)

            # Right-aligned: output frames correspond to the rightmost part
            end_sample = start + chunk_samples
            end_frame = end_sample // dec
            frame_start = end_frame - out_len
            lo = max(0, frame_start)
            hi = min(total_frames, end_frame)
            src_lo = lo - frame_start
            src_hi = src_lo + (hi - lo)
            predictions[lo:hi] = preds[src_lo:src_hi]

    # Fill any remaining frames (if audio shorter than one chunk)
    if np.any(predictions < 0):
        if len(audio_16k) >= chunk_samples:
            start = len(audio_16k) - chunk_samples
            chunk = audio_16k[start: start + chunk_samples]
            x = torch.from_numpy(chunk).float().unsqueeze(0).unsqueeze(0).to(device)
            logits = model(x)
            preds = logits.argmax(dim=1).squeeze(0).cpu().numpy()
            end_frame = len(audio_16k) // dec
            frame_start = end_frame - out_len
            lo = max(0, frame_start)
            hi = min(total_frames, end_frame)
            src_lo = lo - frame_start
            src_hi = src_lo + (hi - lo)
            mask = predictions[lo:hi] < 0
            predictions[lo:hi] = np.where(mask, preds[src_lo:src_hi],
                                          predictions[lo:hi])
        predictions[predictions < 0] = 0

    return predictions


print('VAD_I_C2_L8 + VAD_H_C24_L10 model helpers defined.')

In [ ]:
if DNN_TYPE == 'h':
    # ── VAD_H_C24_L10 raw-waveform model ────────────────────────────────
    model, cfg, device = load_dnn_h_model(DNN_MODEL)
    dnn_labels = dnn_h_predict_file(signal, sr, model, cfg, device)

    dec = cfg.get('decimation_factor', DECIMATION_FACTOR_H)
    target_sr = cfg.get('target_sample_rate', TARGET_SAMPLE_RATE_H)
    dnn_hop_samples_16k = dec                   # in 16 kHz samples
    dnn_hop_s = dnn_hop_samples_16k / target_sr # seconds per frame (4 ms)
elif DNN_TYPE == 'i':
    # ── VAD_I_C2_L8 raw-waveform model ──────────────────────────────────
    model, cfg, device = load_dnn_i_model(DNN_MODEL)
    dnn_labels = dnn_i_predict_file(signal, sr, model, cfg, device)

    dec = cfg.get('decimation_factor', DECIMATION_FACTOR_I)
    target_sr = cfg.get('target_sample_rate', TARGET_SAMPLE_RATE_I)
    dnn_hop_samples_16k = dec                   # in 16 kHz samples
    dnn_hop_s = dnn_hop_samples_16k / target_sr # seconds per frame (8 ms)
else:
    # ── Original mel-feature DNN ─────────────────────────────────────────
    model, cfg, feat_mean, feat_std, device = load_dnn_model(DNN_MODEL)
    dnn_labels = dnn_predict_file(
        signal, sr, model, cfg, feat_mean, feat_std, device
    )
    dnn_hop_samples = cfg.get('hop_samples', int(sr * 0.01))
    dnn_hop_s = dnn_hop_samples / sr

n_dnn = len(dnn_labels)
dnn_times = np.arange(n_dnn) * dnn_hop_s

print(f'DNN model: {n_dnn} frames (arch={cfg["arch"]}, type={DNN_TYPE})')
for lab, name in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
    cnt = np.sum(dnn_labels == lab)
    print(f'  {name:20s}: {cnt:6d} frames  ({cnt/n_dnn*100:5.1f}%)  {cnt*dnn_hop_s:.2f}s')

## 5. Prediction Plots — Waveform + Coloured Timeline

Four panels:
1. **Audio waveform**
2. **Traditional VAD** — coloured timeline (grey=silence, green=single, red=overlap)
3. **DNN model** — coloured timeline
4. **Agreement** between both models (green=agree, red=disagree)

In [ ]:
from matplotlib.patches import Patch
from matplotlib.collections import BrokenBarHCollection

CLASS_COLORS = {0: None, 1: 'cyan', 2: 'orange'}
CLASS_ALPHA  = {0: 0.0, 1: 1.0, 2: 1.0}
CLASS_NAMES  = {0: 'Silence', 1: 'Single speaker', 2: 'Overlap'}


def _label_segments(labels, hop_sec):
    """Convert frame labels to list of (start_sec, duration_sec, label)."""
    segments = []
    if len(labels) == 0:
        return segments
    cur_label = labels[0]
    seg_start = 0.0
    for i in range(1, len(labels)):
        if labels[i] != cur_label:
            segments.append((seg_start, i * hop_sec - seg_start, cur_label))
            cur_label = labels[i]
            seg_start = i * hop_sec
    segments.append((seg_start, len(labels) * hop_sec - seg_start, cur_label))
    return segments


def _draw_timeline(ax, segments, y_bottom=0, height=1):
    """Draw coloured horizontal bars for each segment on *ax*."""
    for start, dur, lab in segments:
        if CLASS_COLORS[lab] is not None:
            ax.barh(y_bottom + height / 2, dur, height=height, left=start,
                    color=CLASS_COLORS[lab], alpha=CLASS_ALPHA[lab],
                    edgecolor='none', linewidth=0)


# ── Build segments ────────────────────────────────────────────────────────
trad_segs = _label_segments(trad_labels, hop_s)
dnn_segs  = _label_segments(dnn_labels,  dnn_hop_s)

# ── Figure ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(22, 13), sharex=True,
                         gridspec_kw={'height_ratios': [1, 1, 1],
                                      'hspace': 0.15})
fig.suptitle(f'Multivoice VAD Predictions — {INPUT_WAV}', fontsize=28, fontweight='bold', y=0.985)

# ── Panel 1: Waveform with GT fill areas ─────────────────────────────────
t_sig = np.arange(len(signal)) / sr

# Ground truth fill colours (very light, behind the waveform)
GT_FILL = {0: (None,    0.0),      # silence  — no fill
           1: ('cyan',  1.0),     # single   — cyan
           2: ('orange', 1.0)}    # overlap  — orange

# Build GT segments from sample-level labels (run-length encoding)
def _gt_sample_segments(gt, sample_rate):
    """Convert sample-level GT to (start_sec, end_sec, label) segments."""
    segs = []
    if len(gt) == 0:
        return segs
    cur = gt[0]; s0 = 0
    for i in range(1, len(gt)):
        if gt[i] != cur:
            segs.append((s0 / sample_rate, i / sample_rate, int(cur)))
            cur = gt[i]; s0 = i
    segs.append((s0 / sample_rate, len(gt) / sample_rate, int(cur)))
    return segs

gt_segs = _gt_sample_segments(gt_vad, sr)
ymin = -1.05 * np.max(np.abs(signal))
ymax =  1.05 * np.max(np.abs(signal))
for t0, t1, lab in gt_segs:
    fc, fa = GT_FILL[lab]
    if fc is not None:
        axes[0].axvspan(t0, t1, color=fc, alpha=fa, zorder=0)

axes[0].plot(t_sig, signal, lw=0.4, color='k', alpha=0.9, zorder=2)
axes[0].set_ylim(ymin, ymax)
axes[0].set_ylabel('Amplitude', fontsize=20)
axes[0].set_title('Audio Waveform (background: manual Ground Truth)', fontsize=16, loc='left')
axes[0].tick_params(axis='both', labelsize=19)
axes[0].margins(x=0)

# ── Panel 2: Traditional VAD ─────────────────────────────────────────────
_draw_timeline(axes[1], trad_segs)
axes[1].set_ylim(0, 1)
axes[1].set_yticks([])
axes[1].set_ylabel('Traditional\nVAD', fontsize=20, fontweight='bold',
                   rotation=0, labelpad=75, va='center')
axes[1].tick_params(axis='x', labelsize=19)
axes[1].margins(x=0)

# ── Panel 3: DNN ─────────────────────────────────────────────────────────
_draw_timeline(axes[2], dnn_segs)
axes[2].set_ylim(0, 1)
axes[2].set_yticks([])
axes[2].set_ylabel('DNN Model\nVAD', fontsize=20, fontweight='bold',
                   rotation=0, labelpad=75, va='center')
axes[2].set_xlabel('Time (s)', fontsize=20)
axes[2].tick_params(axis='x', labelsize=19)
axes[2].margins(x=0)

# ── Shared legend ─────────────────────────────────────────────────────────
legend_patches = [Patch(fc='cyan', ec='none', alpha=1.0, label='Single speaker'),
                  Patch(fc='orange', ec='none', alpha=1.0, label='Overlap')]
fig.legend(handles=legend_patches, loc='upper right', ncol=2,
           framealpha=1.0, bbox_to_anchor=(0.98, 0.96),
           prop={'weight': 'bold', 'size': 20})

fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05)
plt.show()

## 6. Overlap Filtering — 1 s Sliding Window

For each frame labeled as **overlap (2)**, a 1-second window (±0.5 s = ±50 frames at 10 ms hop) is examined.
If **fewer than 50%** of the frames in the window are overlap, the label is changed to **single speaker (1)**.
This removes short/spurious overlap detections while preserving sustained overlap regions.

In [ ]:
# ── Overlap sliding-window filter ─────────────────────────────────────────
def overlap_filter(labels, hop_sec, window_sec=1.0, threshold=0.50):
    """
    For every frame labelled as overlap (2), check a centred window of
    *window_sec* seconds.  If the fraction of overlap frames in the window
    is below *threshold*, re-label that frame as single speaker (1).
    """
    filtered = labels.copy()
    half_win = int((window_sec / 2) / hop_sec)   # frames on each side
    n = len(labels)
    for i in range(n):
        if labels[i] == 2:
            lo = max(0, i - half_win)
            hi = min(n, i + half_win + 1)
            window = labels[lo:hi]
            if np.sum(window == 2) / len(window) < threshold:
                filtered[i] = 1
    return filtered


trad_filtered = overlap_filter(trad_labels, hop_s,     window_sec=1.0, threshold=0.50)
dnn_filtered  = overlap_filter(dnn_labels,  dnn_hop_s, window_sec=1.0, threshold=0.50)

# ── Statistics ────────────────────────────────────────────────────────────
for name, orig, filt, hs in [('Traditional', trad_labels, trad_filtered, hop_s),
                              ('DNN',         dnn_labels,  dnn_filtered, dnn_hop_s)]:
    n_orig_ovl = np.sum(orig == 2)
    n_filt_ovl = np.sum(filt == 2)
    removed = n_orig_ovl - n_filt_ovl
    print(f'{name:12s}  overlap: {n_orig_ovl} → {n_filt_ovl}  '
          f'(removed {removed} frames = {removed * hs:.2f} s)')

# ── Build filtered segments ──────────────────────────────────────────────
trad_filt_segs = _label_segments(trad_filtered, hop_s)
dnn_filt_segs  = _label_segments(dnn_filtered,  dnn_hop_s)

# ── Figure (same layout as Section 5) ────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(22, 13), sharex=True,
                         gridspec_kw={'height_ratios': [1, 1, 1],
                                      'hspace': 0.15})
fig.suptitle(f'Multivoice VAD Predictions (filtered) — {INPUT_WAV}',
             fontsize=28, fontweight='bold', y=0.985)

# ── Panel 1: Waveform with GT ────────────────────────────────────────────
t_sig = np.arange(len(signal)) / sr
gt_segs = _gt_sample_segments(gt_vad, sr)
ymin = -1.05 * np.max(np.abs(signal))
ymax =  1.05 * np.max(np.abs(signal))
for t0, t1, lab in gt_segs:
    fc, fa = GT_FILL[lab]
    if fc is not None:
        axes[0].axvspan(t0, t1, color=fc, alpha=fa, zorder=0)
axes[0].plot(t_sig, signal, lw=0.4, color='k', alpha=0.9, zorder=2)
axes[0].set_ylim(ymin, ymax)
axes[0].set_ylabel('Amplitude', fontsize=20)
axes[0].set_title('Audio Waveform (background: manual Ground Truth)', fontsize=16, loc='left')
axes[0].tick_params(axis='both', labelsize=19)
axes[0].margins(x=0)

# ── Panel 2: Traditional VAD (filtered) ──────────────────────────────────
_draw_timeline(axes[1], trad_filt_segs)
axes[1].set_ylim(0, 1)
axes[1].set_yticks([])
axes[1].set_ylabel('Traditional\nVAD\n(1s window,\n≥50% thr.)', fontsize=20, fontweight='bold',
                   rotation=0, labelpad=90, va='center')
axes[1].tick_params(axis='x', labelsize=19)
axes[1].margins(x=0)

# ── Panel 3: DNN (filtered) ──────────────────────────────────────────────
_draw_timeline(axes[2], dnn_filt_segs)
axes[2].set_ylim(0, 1)
axes[2].set_yticks([])
axes[2].set_ylabel('DNN Model\nVAD\n(1s window,\n≥50% thr.)', fontsize=20, fontweight='bold',
                   rotation=0, labelpad=90, va='center')
axes[2].set_xlabel('Time (s)', fontsize=20)
axes[2].tick_params(axis='x', labelsize=19)
axes[2].margins(x=0)

# ── Shared legend ─────────────────────────────────────────────────────────
legend_patches = [Patch(fc='cyan', ec='none', alpha=1.0, label='Single speaker'),
                  Patch(fc='orange', ec='none', alpha=1.0, label='Overlap')]
fig.legend(handles=legend_patches, loc='upper right', ncol=2,
           framealpha=1.0, bbox_to_anchor=(0.98, 0.96),
           prop={'weight': 'bold', 'size': 20})

fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05)
plt.show()

## 7. Morphological Overlap Filtering — Min Block Width + Gap Filling

Applied on top of the **Section 6 filtered labels** (1 s sliding window, ≥ 50 % threshold).

Two parameters control gap filling between overlap blocks:

1. **Minimum block width** (50 ms): only contiguous blocks of overlap that are **at least** this long qualify as anchors. Shorter blocks are ignored (not used as anchors), but they are **not removed** from the output.
2. **Maximum gap** (300 ms): gaps between qualifying anchor blocks that are ≤ this threshold are filled with overlap.

**Important:** The analysis is performed entirely on the **unmodified input labels** (from Sec.&nbsp;6) — gaps to fill are collected first, and all writes happen only after the full scan is complete, so inserted labels never influence the ongoing analysis.

In [ ]:
# ── Morphological overlap filter ──────────────────────────────────────────
MIN_BLOCK_MS = 50    # minimum width of a contiguous overlap block to keep
MAX_GAP_MS   = 1000   # maximum gap between overlap blocks to fill


def morphological_overlap_filter(labels, hop_sec,
                                  min_block_ms=MIN_BLOCK_MS,
                                  max_gap_ms=MAX_GAP_MS):
    """
    Gap-filling filter for overlap (label 2) predictions.

    Algorithm (read-then-write):
      1. **Analyse** the *original* labels — find all contiguous blocks of 2s.
      2. Keep only blocks whose length ≥ *min_block_ms* as **anchors**.
         (Short blocks are left untouched, they simply don't serve as anchors.)
      3. For every pair of consecutive anchors, check the gap between them.
         If the gap ≤ *max_gap_ms*, **record** it for filling.
      4. After the full scan, fill all recorded gaps with 2s in one pass.

    Returns
    -------
    filtered   : np.ndarray  — same shape as *labels*, with filled gaps.
    n_anchors  : int         — number of qualifying anchor blocks found.
    filled_gap : int         — total number of gap frames filled with 2.
    """
    min_frames     = max(1, int(min_block_ms / 1000 / hop_sec))
    max_gap_frames = max(1, int(max_gap_ms / 1000 / hop_sec))

    # -- Step 1 (read-only): find all contiguous blocks of 2 in ORIGINAL labels
    def _find_blocks(arr, val=2):
        blocks = []
        i = 0
        while i < len(arr):
            if arr[i] == val:
                s = i
                while i < len(arr) and arr[i] == val:
                    i += 1
                blocks.append((s, i))   # [start, end)
            else:
                i += 1
        return blocks

    all_blocks = _find_blocks(labels)

    # -- Step 2 (read-only): keep only anchor blocks (≥ min_frames) --------
    anchors = [(s, e) for s, e in all_blocks if (e - s) >= min_frames]

    # -- Step 3 (read-only): collect gaps between consecutive anchors ------
    gaps_to_fill = []
    for j in range(len(anchors) - 1):
        gap_start = anchors[j][1]       # end of current anchor
        gap_end   = anchors[j + 1][0]   # start of next anchor
        gap_len   = gap_end - gap_start
        if gap_len <= max_gap_frames:
            gaps_to_fill.append((gap_start, gap_end))

    # -- Step 4 (single write pass): fill all recorded gaps ----------------
    filtered = labels.copy()
    filled_gap = 0
    for gs, ge in gaps_to_fill:
        filtered[gs:ge] = 2
        filled_gap += (ge - gs)

    return filtered, len(anchors), filled_gap


# ── Apply to filtered labels from Section 6 ──────────────────────────────
trad_morph, trad_anchors, trad_fill = morphological_overlap_filter(trad_filtered, hop_s)
dnn_morph,  dnn_anchors,  dnn_fill  = morphological_overlap_filter(dnn_filtered, dnn_hop_s)

# ── Statistics ────────────────────────────────────────────────────────────
print(f'Parameters:  min_block ≥ {MIN_BLOCK_MS} ms,  max_gap ≤ {MAX_GAP_MS} ms')
print(f'Input: Section 6 filtered labels (1s sliding window, ≥50% thr.)\n')
for name, filt, morph, n_anch, fill, hs in [
        ('Traditional', trad_filtered, trad_morph, trad_anchors, trad_fill, hop_s),
        ('DNN',         dnn_filtered,  dnn_morph,  dnn_anchors,  dnn_fill, dnn_hop_s)]:
    n_filt  = np.sum(filt == 2)
    n_morph = np.sum(morph == 2)
    print(f'{name:12s}  anchors: {n_anch},  '
          f'overlap: {n_filt} → {n_morph}  '
          f'(filled {fill} gap frames = {fill * hs:.2f} s)')

# ── Build segments ───────────────────────────────────────────────────────
trad_morph_segs = _label_segments(trad_morph, hop_s)
dnn_morph_segs  = _label_segments(dnn_morph,  dnn_hop_s)

# ── Figure (same layout as Section 5/6) ──────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(22, 13), sharex=True,
                         gridspec_kw={'height_ratios': [1, 1, 1],
                                      'hspace': 0.15})
fig.suptitle(
    f'Multivoice VAD Predictions (morph. filtered) — {INPUT_WAV}',
    fontsize=28, fontweight='bold', y=0.985)

# ── Panel 1: Waveform with GT ────────────────────────────────────────────
t_sig = np.arange(len(signal)) / sr
gt_segs = _gt_sample_segments(gt_vad, sr)
ymin = -1.05 * np.max(np.abs(signal))
ymax =  1.05 * np.max(np.abs(signal))
for t0, t1, lab in gt_segs:
    fc, fa = GT_FILL[lab]
    if fc is not None:
        axes[0].axvspan(t0, t1, color=fc, alpha=fa, zorder=0)
axes[0].plot(t_sig, signal, lw=0.4, color='k', alpha=0.9, zorder=2)
axes[0].set_ylim(ymin, ymax)
axes[0].set_ylabel('Amplitude', fontsize=20)
axes[0].set_title('Audio Waveform (background: manual Ground Truth)', fontsize=16, loc='left')
axes[0].tick_params(axis='both', labelsize=19)
axes[0].margins(x=0)

# ── Panel 2: Traditional VAD (morph. filtered) ───────────────────────────
_draw_timeline(axes[1], trad_morph_segs)
axes[1].set_ylim(0, 1)
axes[1].set_yticks([])
axes[1].set_ylabel(f'Traditional\nVAD\n(1s window,\n≥50% thr.)\n[min {MIN_BLOCK_MS}ms,\ngap ≤{MAX_GAP_MS}ms]',
                   fontsize=20, fontweight='bold',
                   rotation=0, labelpad=90, va='center')
axes[1].tick_params(axis='x', labelsize=19)
axes[1].margins(x=0)

# ── Panel 3: DNN (morph. filtered) ───────────────────────────────────────
_draw_timeline(axes[2], dnn_morph_segs)
axes[2].set_ylim(0, 1)
axes[2].set_yticks([])
axes[2].set_ylabel(f'DNN Model\nVAD\n(1s window,\n≥50% thr.)\n[min {MIN_BLOCK_MS}ms,\ngap ≤{MAX_GAP_MS}ms]',
                   fontsize=20, fontweight='bold',
                   rotation=0, labelpad=90, va='center')
axes[2].set_xlabel('Time (s)', fontsize=20)
axes[2].tick_params(axis='x', labelsize=19)
axes[2].margins(x=0)

# ── Shared legend ─────────────────────────────────────────────────────────
legend_patches = [Patch(fc='cyan', ec='none', alpha=1.0, label='Single speaker'),
                  Patch(fc='orange', ec='none', alpha=1.0, label='Overlap')]
fig.legend(handles=legend_patches, loc='upper right', ncol=2,
           framealpha=1.0, bbox_to_anchor=(0.98, 0.96),
           prop={'weight': 'bold', 'size': 20})

fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05)
plt.show()

## 8. Single-Speaker Smoothing + Short Block Removal

Applied on top of the **Section 7** output (overlap-filled labels).

Three operations (in order):

1. **Gap filling for single speaker (label 1):** gaps ≤ 1000 ms between blocks of 1s that are ≥ 100 ms are filled with 1s (same read-then-write approach as Sec. 7).
2. **Short block removal:** after filling, any contiguous speech block (1s or 2s) that is still shorter than 1000 ms is replaced with silence (0).

This removes short spurious speech detections while preserving sustained speech and overlap regions.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────
SINGLE_MIN_BLOCK_MS = 100    # min anchor block width for single-speaker gap fill
SINGLE_MAX_GAP_MS   = 1000   # max gap between single-speaker anchors to fill
MIN_SPEECH_BLOCK_MS = 1000   # speech blocks shorter than this → silence


def gap_fill(labels, hop_sec, target_label, min_block_ms, max_gap_ms):
    """
    Read-then-write gap filler for a specific *target_label*.

    1. (read-only) Find contiguous blocks of *target_label* ≥ *min_block_ms* → anchors.
    2. (read-only) Collect gaps ≤ *max_gap_ms* between consecutive anchors.
    3. (single write) Fill collected gaps with *target_label*.

    Returns  (filtered, n_anchors, filled_frames)
    """
    min_frames     = max(1, int(min_block_ms / 1000 / hop_sec))
    max_gap_frames = max(1, int(max_gap_ms   / 1000 / hop_sec))

    # find contiguous blocks of target_label
    blocks = []
    i = 0
    while i < len(labels):
        if labels[i] == target_label:
            s = i
            while i < len(labels) and labels[i] == target_label:
                i += 1
            blocks.append((s, i))
        else:
            i += 1

    anchors = [(s, e) for s, e in blocks if (e - s) >= min_frames]

    gaps_to_fill = []
    for j in range(len(anchors) - 1):
        gs = anchors[j][1]
        ge = anchors[j + 1][0]
        if (ge - gs) <= max_gap_frames:
            gaps_to_fill.append((gs, ge))

    filtered = labels.copy()
    filled = 0
    for gs, ge in gaps_to_fill:
        filtered[gs:ge] = target_label
        filled += (ge - gs)

    return filtered, len(anchors), filled


def remove_short_speech_blocks(labels, hop_sec, min_speech_ms):
    """
    Replace contiguous speech blocks (labels 1 or 2) shorter than
    *min_speech_ms* with silence (0).

    Returns  (filtered, n_removed_blocks, removed_frames)
    """
    min_frames = max(1, int(min_speech_ms / 1000 / hop_sec))

    # find contiguous speech blocks (runs where label > 0)
    blocks = []
    i = 0
    while i < len(labels):
        if labels[i] > 0:
            s = i
            while i < len(labels) and labels[i] > 0:
                i += 1
            blocks.append((s, i))
        else:
            i += 1

    filtered = labels.copy()
    n_removed = 0
    removed_frames = 0
    for s, e in blocks:
        if (e - s) < min_frames:
            filtered[s:e] = 0
            n_removed += 1
            removed_frames += (e - s)

    return filtered, n_removed, removed_frames


# ── Step A: Fill single-speaker gaps (on Section 7 output) ───────────────
trad_s8a, trad_s_anch, trad_s_fill = gap_fill(
    trad_morph, hop_s, target_label=1,
    min_block_ms=SINGLE_MIN_BLOCK_MS, max_gap_ms=SINGLE_MAX_GAP_MS)
dnn_s8a, dnn_s_anch, dnn_s_fill = gap_fill(
    dnn_morph, dnn_hop_s, target_label=1,
    min_block_ms=SINGLE_MIN_BLOCK_MS, max_gap_ms=SINGLE_MAX_GAP_MS)

# ── Step B: Remove short speech blocks ───────────────────────────────────
trad_s8, trad_n_rm, trad_rm_fr = remove_short_speech_blocks(
    trad_s8a, hop_s, MIN_SPEECH_BLOCK_MS)
dnn_s8, dnn_n_rm, dnn_rm_fr = remove_short_speech_blocks(
    dnn_s8a, dnn_hop_s, MIN_SPEECH_BLOCK_MS)

# ── Statistics ────────────────────────────────────────────────────────────
print(f'Single-speaker gap fill:  min_block ≥ {SINGLE_MIN_BLOCK_MS} ms,  '
      f'max_gap ≤ {SINGLE_MAX_GAP_MS} ms')
print(f'Short block removal:      min_speech ≥ {MIN_SPEECH_BLOCK_MS} ms\n')

for name, morph, s8a, s8, s_anch, s_fill, n_rm, rm_fr, hs in [
        ('Traditional', trad_morph, trad_s8a, trad_s8,
         trad_s_anch, trad_s_fill, trad_n_rm, trad_rm_fr, hop_s),
        ('DNN', dnn_morph, dnn_s8a, dnn_s8,
         dnn_s_anch, dnn_s_fill, dnn_n_rm, dnn_rm_fr, dnn_hop_s)]:
    print(f'{name:12s}')
    n1_before = np.sum(morph == 1)
    n1_after  = np.sum(s8a == 1)
    print(f'  gap fill : anchors={s_anch}, '
          f'single: {n1_before} → {n1_after}  '
          f'(filled {s_fill} frames = {s_fill * hs:.2f} s)')
    for lab, nm in [(0, 'silence'), (1, 'single'), (2, 'overlap')]:
        c_before = np.sum(s8a == lab)
        c_after  = np.sum(s8 == lab)
        diff = c_after - c_before
        if diff != 0:
            print(f'  cleanup  : {nm}: {c_before} → {c_after}  '
                  f'({diff:+d} frames = {diff * hs:+.2f} s)')
    print(f'  removed  : {n_rm} short blocks ({rm_fr} frames = {rm_fr * hs:.2f} s)')
    print()

# ── Build segments ───────────────────────────────────────────────────────
trad_s8_segs = _label_segments(trad_s8, hop_s)
dnn_s8_segs  = _label_segments(dnn_s8,  dnn_hop_s)

# ── Figure ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(22, 13), sharex=True,
                         gridspec_kw={'height_ratios': [1, 1, 1],
                                      'hspace': 0.15})
fig.suptitle(
    f'Multivoice VAD Predictions (smoothed) — {INPUT_WAV}',
    fontsize=28, fontweight='bold', y=0.985)

# ── Panel 1: Waveform with GT ────────────────────────────────────────────
t_sig = np.arange(len(signal)) / sr
gt_segs = _gt_sample_segments(gt_vad, sr)
ymin = -1.05 * np.max(np.abs(signal))
ymax =  1.05 * np.max(np.abs(signal))
for t0, t1, lab in gt_segs:
    fc, fa = GT_FILL[lab]
    if fc is not None:
        axes[0].axvspan(t0, t1, color=fc, alpha=fa, zorder=0)
axes[0].plot(t_sig, signal, lw=0.4, color='k', alpha=0.9, zorder=2)
axes[0].set_ylim(ymin, ymax)
axes[0].set_ylabel('Amplitude', fontsize=20)
axes[0].set_title('Audio Waveform (background: manual Ground Truth)',
                  fontsize=16, loc='left')
axes[0].tick_params(axis='both', labelsize=19)
axes[0].margins(x=0)

# ── Panel 2: Traditional VAD (smoothed) ──────────────────────────────────
_draw_timeline(axes[1], trad_s8_segs)
axes[1].set_ylim(0, 1)
axes[1].set_yticks([])
axes[1].set_ylabel(f'Traditional\nVAD\n(1s window,\n≥50% thr.)\n[min {MIN_BLOCK_MS}ms,\ngap ≤{MAX_GAP_MS}ms]\n<smoothed,\n<{MIN_SPEECH_BLOCK_MS}ms→sil>',
                   fontsize=20, fontweight='bold',
                   rotation=0, labelpad=90, va='center')
axes[1].tick_params(axis='x', labelsize=19)
axes[1].margins(x=0)

# ── Panel 3: DNN (smoothed) ─────────────────────────────────────────────
_draw_timeline(axes[2], dnn_s8_segs)
axes[2].set_ylim(0, 1)
axes[2].set_yticks([])
axes[2].set_ylabel(f'DNN Model\nVAD\n(1s window,\n≥50% thr.)\n[min {MIN_BLOCK_MS}ms,\ngap ≤{MAX_GAP_MS}ms]\n<smoothed,\n<{MIN_SPEECH_BLOCK_MS}ms→sil>',
                   fontsize=20, fontweight='bold',
                   rotation=0, labelpad=90, va='center')
axes[2].set_xlabel('Time (s)', fontsize=20)
axes[2].tick_params(axis='x', labelsize=19)
axes[2].margins(x=0)

# ── Shared legend ─────────────────────────────────────────────────────────
legend_patches = [Patch(fc='cyan', ec='none', alpha=1.0, label='Single speaker'),
                  Patch(fc='orange', ec='none', alpha=1.0, label='Overlap')]
fig.legend(handles=legend_patches, loc='upper right', ncol=2,
           framealpha=1.0, bbox_to_anchor=(0.98, 0.96),
           prop={'weight': 'bold', 'size': 20})

fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05)
plt.show()